In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
from tqdm import tqdm

save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/owen_reservoirs/Columbia")
metadata_dir = save_dir / "metadata"

subbasins = pd.read_parquet(metadata_dir / 'subbasins_4km2.parquet', columns=['site_id', 'nextdown','outlet_id'])
subbasins.index = subbasins.index.astype(str)
subbasins.reset_index(inplace=True)

subbasins

,site_id,nextdown,outlet_id
0,78026418,7820017443,USGS-14246900
1,78014072,7820034213,USGS-14246900
2,78015448,7820039453,USGS-14246900
3,78019645,7820020163,USGS-14246900
4,78021658,7820052763,USGS-14246900
...,...,...,...
2139,7820069173,USGS-12433000,USGS-14246900
2140,7820071213,7820071293,USGS-14246900
2141,7820071293,78008362,USGS-14246900
2142,7820077403,78011914,USGS-14246900


In [3]:
mask = subbasins['nextdown'].isna()

# mapping of old → new
mapping = subbasins.loc[mask].set_index('site_id')['outlet_id']

# Step 1: update site_id
subbasins.loc[mask, 'site_id'] = subbasins.loc[mask, 'outlet_id']

# Step 2: update nextdown using the mapping
subbasins['nextdown'] = subbasins['nextdown'].replace(mapping.to_dict())

In [4]:
G = nx.DiGraph()

# Add all subbasin IDs as nodes
G.add_nodes_from(subbasins['site_id'])

# Add edges for those with valid nextdown entries
edges = list(subbasins.loc[subbasins['nextdown'].notna(), ['site_id','nextdown']].itertuples(index=False, name=None))
G.add_edges_from(edges)

In [5]:
outlets = set(subbasins['outlet_id'].unique())
missing = []
for nodes in nx.weakly_connected_components(G):
    subG = G.subgraph(nodes).copy()

    # Find outlet: node with no outgoing edges
    outlet = [n for n in subG.nodes if subG.out_degree(n) == 0]
    if len(outlet) != 1:
        raise ValueError(f"Expected exactly one outlet per basin, found {outlets}")
    basin_id = outlet[0]
    
    outlets.remove(basin_id)
        
outlets

set()

In [6]:
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

# Find sources and sinks
sources = [n for n in G.nodes() if G.in_degree(n) == 0]
sinks = [n for n in G.nodes() if G.out_degree(n) == 0]
print(f"Sources (headwaters): {len(sources)}")
print(f"Sinks (outlets): {len(sinks)}")

Nodes: 2144
Edges: 2143
Sources (headwaters): 863
Sinks (outlets): 1


In [7]:
import json

graph_json = nx.readwrite.json_graph.node_link_data(G, edges='edges')
with open(metadata_dir / f'graphs_4km2.json', 'w') as f:
    json.dump(graph_json, f, indent=4)